# Clasificador KNN con 10,000 registros

Este ejemplo utiliza el dataset **Adult** del repositorio UCI. El objetivo es clasificar si el ingreso anual de una persona es menor o igual a 50,000 dolares (`<=50K`) o mayor a 50,000 dolares (`>50K`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 1. Carga del dataset

Se descargan los datos originales de UCI y se asignan sus nombres de columnas.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columnas = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]
dataset = pd.read_csv(url, names=columnas, sep=r",\s*", engine="python", na_values="?")

dataset = dataset.dropna()
dataset["income"] = dataset["income"].str.strip().str.replace(".", "", regex=False)

print("Registros originales disponibles:", len(dataset))
dataset.head()

## 2. Seleccion de exactamente 10,000 registros

In [ ]:
data = dataset.sample(n=10000, random_state=42)

print("Forma del dataset seleccionado:", data.shape)
print("Registros:", len(data))
print("\nDistribucion de clases:")
print(data["income"].value_counts())

In [ ]:
X = data.drop(columns="income")
y = data["income"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

print("Variables numericas:", list(numeric_features))
print("Variables categoricas:", list(categorical_features))

## 3. Division de datos y preprocesamiento

El 80% se utiliza para entrenar y el 20% para probar. Las variables numericas se estandarizan y las categoricas se convierten a variables binarias.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

## 4. Entrenamiento del clasificador KNN

In [ ]:
knn = Pipeline(steps=[
    ("preprocesamiento", preprocessor),
    ("clasificador", KNeighborsClassifier(n_neighbors=5, weights="distance"))
])

knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

## 5. Evaluacion del modelo

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Precision (accuracy):", round(accuracy * 100, 2), "%")
print("\nReporte de clasificacion:")
print(classification_report(y_test, y_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Matriz de confusion - Clasificador KNN")
plt.tight_layout()
plt.show()

## 6. Comparacion de diferentes valores de k

In [ ]:
valores_k = [3, 5, 7, 9]
precisiones = []

for k in valores_k:
    modelo = Pipeline(steps=[
        ("preprocesamiento", preprocessor),
        ("clasificador", KNeighborsClassifier(n_neighbors=k, weights="distance"))
    ])
    modelo.fit(X_train, y_train)
    prediccion = modelo.predict(X_test)
    precision = accuracy_score(y_test, prediccion)
    precisiones.append(precision)
    print(f"k={k}: accuracy={precision:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(valores_k, precisiones, marker="o")
plt.xlabel("Numero de vecinos (k)")
plt.ylabel("Accuracy")
plt.title("Comparacion de valores de k")
plt.grid(True)
plt.xticks(valores_k)
plt.show()